# Lesson 14 Lab — bitsandbytes 4-Bit Loading: NF4, Compute Dtype, and Nested Quantization

**Puzzle:** Does `load_in_4bit=True` specify how the layer computes?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

`load_in_4bit=True` is not a complete numerical specification. A bitsandbytes configuration also selects a codebook such as NF4, a compute dtype for dequantized matrix operations, and optionally nested quantization for metadata. The loaded module class and backend availability determine whether those settings became a real operator or stayed configuration text.


## 0. Predict before running

1. Distinguish quantization codebook, packed storage dtype, compute dtype, and nested quantization.
2. Predict whether NF4 or uniform INT4 gives lower RMSE for normally distributed weights.
3. State what evidence would be required to label the result a native bitsandbytes run.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

A bitsandbytes 4-bit configuration contains at least storage codebook (`NF4` or FP4), compute dtype, optional double/nested quantization, and the module/backend that consumes it.

- Storage type, quantization codebook, and compute dtype are separate choices.
- Nested quantization compresses quantization metadata; it does not turn activation compute into two-bit arithmetic.
- Package presence and device support must be checked before claiming a bitsandbytes run.


## 2. Derive the mechanism

NF4 assigns its 16 codes non-uniformly rather than at equal integer spacing. During a linear operation the packed codes are dequantized or consumed by a fused path while activations use the configured compute dtype.

Uniform INT4 places evenly spaced reconstruction levels over a selected range. NF4 instead uses a non-uniform codebook whose levels allocate more resolution where a normal distribution has more probability mass. A stored code selects one level; matrix multiplication still needs dequantization/scaling and a floating-point compute path. Double or nested quantization reduces the cost of quantization constants, not the activation arithmetic to two bits.

Codebook quality depends on the weight distribution and normalization rule. NF4 can lower average error for bell-shaped weights while producing a larger worst-case error near tails than a range-fitted uniform grid. The deployment decision also includes kernel support and compute dtype stability.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "14-bitsandbytes-4bit"
device = require_cuda()
torch.manual_seed(2026 + 14)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | uniform symmetric INT4 reconstruction of normally distributed weights |
| Candidate | reference NF4 codebook reconstruction of the same weights |
| Held constant | weight tensor, normalization, number of codes, error reference, seed |
| Measurements | RMSE/MAE/cosine/max error and bitsandbytes installation probe |
| Evidence | `numerical-model` |

**Experiment:** Compare a reference NF4 codebook with uniform INT4 on normally distributed weights and probe whether bitsandbytes is installed.


## 5. Read the experiment code

The lab isolates codebook reconstruction and separately records package presence, so a numerical NF4 result cannot masquerade as bitsandbytes execution.

The notebook maps the same random weights through a reference NF4 codebook and a uniform INT4 quantizer, then compares both with the original tensor. It separately checks whether bitsandbytes is importable. Keeping these branches separate prevents a numerical codebook experiment from masquerading as a library benchmark.

No transformers model is loaded, no `Linear4bit` module is instantiated, and no bitsandbytes kernel is timed in the checked-in environment. The evidence label therefore remains `numerical-model`.

Only after these variables match the protocol should the cell be executed.


In [2]:
import importlib.util
nf4=torch.tensor([-1.0,-0.6962,-0.5251,-0.3949,-0.2844,-0.1848,-0.0911,0.0,0.0796,0.1609,0.2461,0.3379,0.4407,0.5626,0.7230,1.0],device=device)
w=torch.randn(1_000_000,device=device); scale=w.abs().max(); normalized=(w/scale).clamp(-1,1)
idx=(normalized[:,None]-nf4[None,:]).abs().argmin(1); nf4_dq=nf4[idx]*scale
_,_,uniform=symmetric_quantize(w.reshape(1000,1000),bits=4,group_size=1000); uniform=uniform.reshape(-1)
result=base_result(14,"numerical-model"); result.update({"bitsandbytes_installed":importlib.util.find_spec("bitsandbytes") is not None,
    "nf4_error":error_metrics(w,nf4_dq),"uniform_int4_error":error_metrics(w,uniform),
    "conclusion":"Codebook behavior was measured numerically; bitsandbytes native execution is claimed only when installed."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| NF4 RMSE | 0.127836 |
| Uniform INT4 RMSE | 0.142396 |
| NF4 max error | 0.719360 |
| Uniform INT4 max error | 0.339060 |
| bitsandbytes installed | no |


## 7. Interpret rather than merely print

NF4 achieved RMSE 0.127836 and MAE 0.109566, lower than uniform INT4 at RMSE 0.142396 and MAE 0.122676 for this normal tensor. Uniform INT4 had a smaller max error, 0.339059 versus NF4's 0.719360, showing that average and tail objectives can disagree. The environment probe reported `bitsandbytes_installed=false`.

The result supports the distribution-aware codebook intuition only. It says nothing about native layer memory, throughput, nested-quant overhead, or task quality on this RTX stack.

**Inspection rule:** The numerical comparison explains codebooks. Only an installed bitsandbytes layer would support a native-backend claim.


## 8. Keep the evidence label honest

This run is labeled **`numerical-model`**. The CUDA numerical experiment isolates an algorithmic mechanism. It is not the paper's complete implementation and does not establish a production kernel speedup.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "bitsandbytes_installed": false,
  "conclusion": "Codebook behavior was measured numerically; bitsandbytes native execution is claimed only when installed.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "numerical-model",
  "executed_at_utc": "2026-08-07T14:45:42+00:00",
  "lesson": 14,
  "nf4_error": {
    "cosine": 0.99187696,
    "mae": 0.10956612,
    "max_abs": 0.71935964,
    "rmse": 0.12783626
  },
  "schema_version": 1,
  "uniform_int4_error": {
    "cosine": 0.99001992,
    "mae": 0.12267612,
    "max_abs": 0.33905974,
    "rmse": 0.14239575
  }
}
Saved: artifacts/rtx5090-result.json


## 9. Make the bounded decision

> Record quantization type, compute dtype, nested-quant setting, and actual module class together.

**Acceptance/rollback:** Capture `BitsAndBytesConfig`, package/CUDA compatibility, actual module class, storage bytes, operator evidence, output regression, and timing.

**Failure analysis:** A reference codebook can differ from library normalization, block size, packing, and scale dtype. Claiming bitsandbytes speed from it would be false. Another trap is choosing NF4 from average RMSE while a downstream layer is sensitive to rare tail errors.


## 10. Extend the evidence

Install a release compatible with the current PyTorch/CUDA stack, load one `Linear4bit` layer, and record its actual module, storage tensors, compute dtype, output error, and operator trace. Repeat with and without nested quantization and then with a small model-quality suite.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
